# LeetCode 4109 — Count Valid Suffix Sorts II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def countSuffix(self, nums: List[int], target: List[int]) -> List[int]:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4109 (Weekly Contest 527 — câu 4, rating khoảng 2384).

Đây là phiên bản khó của bài **4107 — Count Valid Suffix Sorts I** (đã có notebook trong thư mục `medium/`). Bài I hỏi về **một** mảng `target`; bài II hỏi về **mọi cửa sổ** độ dài `n` trượt trên một mảng `target` dài hơn. Nếu làm lại bài I cho từng cửa sổ thì quá chậm, nên ta cần thêm ba công cụ:

1. **Cửa sổ trượt với bảng đếm (sliding multiset)** — kiểm tra "cùng đa tập" trong `O(1)` mỗi bước.
2. **Mảng "chỗ đi lên gần nhất"** — tính phần "hậu tố không tăng" cho mọi cửa sổ.
3. **Hàm Z (Z-function)** — tính độ dài tiền tố chung dài nhất giữa `nums` và mọi cửa sổ trong `O(n + m)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Nhắc lại lời giải bài I: đáp án là độ dài khoảng `[minI, maxI]` (nếu cùng đa tập).
2. Brute Force và cách "chuẩn bị trước kết quả rồi tra bảng".
3. Áp dụng bài I cho từng cửa sổ: `O(n · m)`.
4. Ba công cụ trên, ghép thành lời giải `O(n + m)`.
5. Hàm Z từ con số 0: định nghĩa, cách tính ngây thơ, cách tính nhanh với "hộp Z".
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai mảng số nguyên `nums` (độ dài `n`) và `target` (độ dài `m`), với `n <= m`.

Với mỗi chỉ số `i` từ `0` tới `n - 1`, ta có một **thí nghiệm**: giữ nguyên `nums[0..i-1]`, **sắp xếp không tăng** (giảm dần) hậu tố `nums[i..n-1]`. Gọi kết quả là `R_i`.

Trả về mảng `ans` độ dài `m - n + 1`, trong đó:

```text
ans[k] = số chỉ số i sao cho R_i == target[k .. k + n - 1]
```

(tức là đếm số thí nghiệm khớp với **cửa sổ** độ dài `n` bắt đầu tại `k`).

---

## Ví dụ 1

```python
nums = [2, 2, 3], target = [2, 2, 3, 2, 1, 3, 2, 2]   ->   [1, 1, 0, 0, 0, 1]
```

Ba thí nghiệm: `R_0 = [3, 2, 2]`, `R_1 = [2, 3, 2]`, `R_2 = [2, 2, 3]`.

| k | cửa sổ | khớp với | ans[k] |
|---:|---|---|---:|
| 0 | `[2, 2, 3]` | `R_2` | 1 |
| 1 | `[2, 3, 2]` | `R_1` | 1 |
| 2 | `[3, 2, 1]` | — | 0 |
| 3 | `[2, 1, 3]` | — | 0 |
| 4 | `[1, 3, 2]` | — | 0 |
| 5 | `[3, 2, 2]` | `R_0` | 1 |

## Ví dụ 2

```python
nums = [5, 6, 5], target = [5, 6, 5, 5, 6, 5]   ->   [2, 1, 0, 2]
```

`R_0 = [6, 5, 5]`, còn `R_1 = R_2 = [5, 6, 5]` (hậu tố đã giảm dần sẵn). Cửa sổ `[5, 6, 5]` khớp **hai** thí nghiệm nên được đếm `2`.

---

## Ràng buộc

```text
1 <= n <= m <= 10^5
1 <= nums[i], target[i] <= 10^5
```

# 2. Nhắc lại bài I (4107)

Với **một** cửa sổ `W` cùng độ dài với `nums`, bài I chứng minh:

> Thí nghiệm `i` cho ra `W` ⇔ (A) `nums[:i] == W[:i]`, (B) `nums` và `W` **cùng đa tập**, (C) `W[i:]` **không tăng**.

Và rút gọn thành:

```text
nếu không cùng đa tập:  0
ngược lại:              max(0, maxI - minI + 1)
maxI = độ dài tiền tố chung của nums và W  (chặn trên ở n - 1)
minI = vị trí "đi lên" cuối cùng của W     (W[j-1] < W[j]), hoặc 0 nếu không có
```

(Độ dài tiền tố chung = chỉ số khác nhau đầu tiên; nếu giống hệt thì bằng `n`, ta chặn về `n - 1` vì `i <= n - 1`.)

In [1]:
from collections import Counter
from typing import List


def solve_one_window(nums: List[int], W: List[int]) -> int:
    n = len(nums)
    if Counter(nums) != Counter(W):
        return 0
    lcp = 0
    while lcp < n and nums[lcp] == W[lcp]:
        lcp += 1
    max_i = min(lcp, n - 1)
    min_i = 0
    for j in range(n - 1, 0, -1):
        if W[j - 1] < W[j]:
            min_i = j
            break
    return max(0, max_i - min_i + 1)


print(solve_one_window([2, 2, 3], [2, 2, 3]))   # 1
print(solve_one_window([5, 6, 5], [5, 6, 5]))   # 2

1
2


Bài II chỉ là: **tính ba đại lượng "cùng đa tập", `maxI`, `minI` cho mọi cửa sổ** — nhưng phải nhanh.

---

# 3. Cách 1 — Brute Force

Làm thật mọi thí nghiệm cho mọi cửa sổ.

In [2]:
def count_suffix_bruteforce(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    ans = []
    for k in range(m - n + 1):
        window = target[k:k + n]
        cnt = 0
        for i in range(n):
            if nums[:i] + sorted(nums[i:], reverse=True) == window:
                cnt += 1
        ans.append(cnt)
    return ans


print(count_suffix_bruteforce([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))   # [1, 1, 0, 0, 0, 1]
print(count_suffix_bruteforce([5, 6, 5], [5, 6, 5, 5, 6, 5]))         # [2, 1, 0, 2]

[1, 1, 0, 0, 0, 1]
[2, 1, 0, 2]


Độ phức tạp `O((m - n + 1) · n² log n)` — vô vọng với `10^5`.

# 4. Cách 2 — Chuẩn bị trước các kết quả, rồi tra bảng

Các thí nghiệm `R_0, ..., R_{n-1}` **không phụ thuộc cửa sổ**. Tính chúng **một lần**, đếm bằng `Counter` với khóa là `tuple` (list không làm khóa được vì có thể thay đổi). Mỗi cửa sổ chỉ cần tra `Counter[tuple(window)]`.

In [3]:
def count_suffix_lookup(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    results = Counter(tuple(nums[:i] + sorted(nums[i:], reverse=True)) for i in range(n))
    return [results[tuple(target[k:k + n])] for k in range(m - n + 1)]


print(count_suffix_lookup([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))
print(count_suffix_lookup([5, 6, 5], [5, 6, 5, 5, 6, 5]))

[1, 1, 0, 0, 0, 1]
[2, 1, 0, 2]


Chuẩn bị tốn `O(n² log n)`, mỗi cửa sổ tốn `O(n)` để tạo `tuple` → tổng `O(n² log n + m·n)`. Vẫn quá chậm khi `n` lớn, nhưng đây là một kỹ thuật hay: **tách phần không phụ thuộc truy vấn ra làm trước**.

# 5. Cách 3 — Áp dụng bài I cho từng cửa sổ: O(n · m)

Dùng `solve_one_window` cho mỗi cửa sổ: mỗi lần `O(n)`.

In [4]:
def count_suffix_per_window(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    return [solve_one_window(nums, target[k:k + n]) for k in range(m - n + 1)]


print(count_suffix_per_window([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))
print(count_suffix_per_window([5, 6, 5], [5, 6, 5, 5, 6, 5]))

[1, 1, 0, 0, 0, 1]
[2, 1, 0, 2]


Với `n = m/2 = 5·10^4`, số phép toán khoảng `n · (m - n) ≈ 2.5·10^9` — vẫn TLE. Ta cần tính ba đại lượng cho **tất cả** cửa sổ mà mỗi cửa sổ chỉ tốn `O(1)`.

---

# 6. Công cụ 1 — Cửa sổ trượt kiểm tra "cùng đa tập"

Giữ một bảng `cnt = Counter(nums) - Counter(cửa sổ hiện tại)` (cho phép giá trị âm) và một biến `diff` = **số khóa có `cnt` khác 0**. Cùng đa tập ⇔ `diff == 0`.

Khi cửa sổ trượt sang phải một bước: một phần tử **vào** (giảm `cnt` của nó đi 1), một phần tử **ra** (tăng `cnt` của nó lên 1). Mỗi lần thay đổi một khóa, ta cập nhật `diff` tùy theo `cnt` vừa **trở thành 0** hay **vừa rời khỏi 0**.

In [5]:
def same_multiset_flags(nums: List[int], target: List[int]) -> List[bool]:
    n, m = len(nums), len(target)
    cnt = Counter(nums)
    diff = len(cnt)                       # số khóa có cnt != 0

    def change(v: int, d: int):
        nonlocal diff
        before = cnt[v]
        cnt[v] += d
        if before == 0:
            diff += 1                     # vừa rời khỏi 0
        elif cnt[v] == 0:
            diff -= 1                     # vừa trở thành 0

    flags = []
    for i, v in enumerate(target):
        change(v, -1)                     # target[i] vào cửa sổ
        s = i - n + 1                     # đầu cửa sổ
        if s < 0:
            continue
        flags.append(diff == 0)
        change(target[s], +1)             # target[s] ra khỏi cửa sổ (chuẩn bị bước sau)
    return flags


print(same_multiset_flags([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))

[True, True, False, False, False, True]


Cửa sổ 0 `[2,2,3]`, cửa sổ 1 `[2,3,2]`, cửa sổ 5 `[3,2,2]` cùng đa tập với `nums` → `True`; các cửa sổ chứa số `1` thì không.

# 7. Công cụ 2 — Mảng "chỗ đi lên gần nhất"

`minI` của cửa sổ `[s, e]` là vị trí đi lên cuối cùng **bên trong** cửa sổ (tính tương đối). Định nghĩa trên toàn `target`:

```text
prefix[0] = 0
prefix[j] = j            nếu target[j-1] < target[j]   (j là một chỗ đi lên)
prefix[j] = prefix[j-1]  ngược lại
```

`prefix[e]` = chỗ đi lên gần nhất `<= e`. Nếu `prefix[e] > s`, chỗ đi lên đó nằm trong cửa sổ → `minI = prefix[e] - s`. Nếu `prefix[e] <= s`, không có chỗ đi lên nào **bên trong** (chỗ đi lên tại đúng `s` liên quan tới `target[s-1]` nằm ngoài cửa sổ) → `minI = 0`. Gộp lại:

```text
minI = max(prefix[e] - s, 0)
```

In [6]:
target = [2, 2, 3, 2, 1, 3, 2, 2]
m = len(target)
prefix = [0] * m
for j in range(1, m):
    prefix[j] = j if target[j - 1] < target[j] else prefix[j - 1]
print("target =", target)
print("prefix =", prefix)

target = [2, 2, 3, 2, 1, 3, 2, 2]
prefix = [0, 0, 2, 2, 2, 5, 5, 5]


Các chỗ đi lên của `target` là `j = 2` (`2 < 3`) và `j = 5` (`1 < 3`).

# 8. Công cụ 3 — Hàm Z (Z-function)

`maxI` cần **độ dài tiền tố chung dài nhất (LCP)** giữa `nums` và cửa sổ bắt đầu tại `s`, cho mọi `s`. Hàm Z làm đúng việc này.

## 8.1. Định nghĩa

Với một dãy `S` độ dài `L`:

```text
z[i] = độ dài đoạn dài nhất bắt đầu tại i mà trùng với tiền tố của S
     = LCP(S, S[i:])
```

Quy ước `z[0] = L`.

Ví dụ `S = "aabxaab"`:

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| S[i] | a | a | b | x | a | a | b |
| z[i] | 7 | 1 | 0 | 0 | 3 | 1 | 0 |

`z[4] = 3` vì `S[4:] = "aab"` trùng với tiền tố `"aab"`.

## 8.2. Cách tính ngây thơ O(L²)

In [7]:
def z_naive(S) -> List[int]:
    L = len(S)
    z = [0] * L
    if L:
        z[0] = L
    for i in range(1, L):
        while i + z[i] < L and S[z[i]] == S[i + z[i]]:
            z[i] += 1
    return z


print(z_naive("aabxaab"))

[7, 1, 0, 0, 3, 1, 0]


## 8.3. Cách tính nhanh O(L) với "hộp Z"

Duy trì đoạn `[l, r)` có `r` lớn nhất đã biết sao cho `S[l:r]` trùng với tiền tố `S[0:r-l]`. Khi xét `i < r`:

- `i` nằm trong hộp, nên `S[i:r]` trùng với `S[i-l : r-l]` — một đoạn ta **đã tính** `z` rồi.
- Do đó `z[i]` ít nhất là `min(r - i, z[i - l])` — ta **bắt đầu so sánh từ đó**, không phải từ 0.
- Sau đó mới kéo dài thủ công. Nếu `i + z[i] > r`, cập nhật hộp `[l, r) = [i, i + z[i])`.

Mỗi lần kéo dài thủ công làm `r` tăng, mà `r` không vượt `L` → tổng `O(L)`.

In [8]:
def z_function(S) -> List[int]:
    L = len(S)
    z = [0] * L
    if L:
        z[0] = L
    l = r = 0
    for i in range(1, L):
        if i < r:
            z[i] = min(r - i, z[i - l])          # tận dụng thông tin trong hộp
        while i + z[i] < L and S[z[i]] == S[i + z[i]]:
            z[i] += 1                            # kéo dài thủ công
        if i + z[i] > r:
            l, r = i, i + z[i]                   # cập nhật hộp
    return z


print(z_function("aabxaab"))
import random
random.seed(1)
for _ in range(500):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 15)))
    assert z_function(s) == z_naive(s)
print("z_function khớp z_naive trên 500 chuỗi ngẫu nhiên")

[7, 1, 0, 0, 3, 1, 0]
z_function khớp z_naive trên 500 chuỗi ngẫu nhiên


## 8.4. Dùng hàm Z để tính LCP giữa `nums` và mọi vị trí của `target`

Ghép `S = nums + [0] + target`. Số `0` là **lính canh**: không có giá trị nào trong đề bằng `0` (vì `nums[i], target[i] >= 1`), nên mọi đoạn trùng khớp không thể vượt qua nó.

Khi đó với cửa sổ bắt đầu tại `s` trong `target`, vị trí tương ứng trong `S` là `n + 1 + s`, và:

```text
LCP(nums, target[s:]) = z[n + 1 + s]       (không vượt quá n nhờ lính canh)
maxI = min(z[n + 1 + s], n - 1)
```

In [9]:
nums = [2, 2, 3]
target = [2, 2, 3, 2, 1, 3, 2, 2]
n = len(nums)
z = z_function(nums + [0] + target)
print("LCP(nums, target[s:]) cho s = 0..m-1:", z[n + 1:])

LCP(nums, target[s:]) cho s = 0..m-1: [3, 1, 0, 1, 0, 0, 2, 1]


# 9. Cách 4 — Ghép ba công cụ: O(n + m)

Với mỗi cửa sổ `[s, e]` (`e = s + n - 1`):

```text
nếu không cùng đa tập:   ans = 0
ngược lại:               ans = max(0, min(z[n+1+s], n-1) - max(prefix[e] - s, 0) + 1)
```

## Mô phỏng ví dụ 1

`z[n+1:] = [3, 1, 0, 1, 0, 0, 2, 1]`, `prefix = [0, 0, 2, 2, 2, 5, 5, 5]`.

| s | cửa sổ | cùng đa tập? | LCP | maxI | prefix[e] | minI | ans |
|---:|---|:---:|---:|---:|---:|---:|---:|
| 0 | `[2,2,3]` | ✔ | 3 | 2 | 2 | 2 | 1 |
| 1 | `[2,3,2]` | ✔ | 1 | 1 | 2 | 1 | 1 |
| 2 | `[3,2,1]` | ✗ | 0 | 0 | 2 | 0 | 0 |
| 3 | `[2,1,3]` | ✗ | 1 | 1 | 5 | 2 | 0 |
| 4 | `[1,3,2]` | ✗ | 0 | 0 | 5 | 1 | 0 |
| 5 | `[3,2,2]` | ✔ | 0 | 0 | 5 | 0 | 1 |

Kết quả `[1, 1, 0, 0, 0, 1]` ✔.

In [10]:
def count_suffix_fast(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    z = z_function(nums + [0] + target)

    prefix = [0] * m
    for j in range(1, m):
        prefix[j] = j if target[j - 1] < target[j] else prefix[j - 1]

    cnt = Counter(nums)
    diff = len(cnt)
    ans = []
    for i, v in enumerate(target):
        # target[i] vào cửa sổ
        before = cnt[v]; cnt[v] -= 1
        diff += 1 if before == 0 else (-1 if cnt[v] == 0 else 0)
        s = i - n + 1
        if s < 0:
            continue
        if diff == 0:
            max_i = min(z[n + 1 + s], n - 1)
            min_i = max(prefix[i] - s, 0)
            ans.append(max(0, max_i - min_i + 1))
        else:
            ans.append(0)
        # target[s] ra khỏi cửa sổ
        w = target[s]
        before = cnt[w]; cnt[w] += 1
        diff += 1 if before == 0 else (-1 if cnt[w] == 0 else 0)
    return ans


print(count_suffix_fast([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]))   # [1, 1, 0, 0, 0, 1]
print(count_suffix_fast([5, 6, 5], [5, 6, 5, 5, 6, 5]))         # [2, 1, 0, 2]

[1, 1, 0, 0, 0, 1]
[2, 1, 0, 2]


In [11]:
def trace_fast(nums: List[int], target: List[int]):
    n, m = len(nums), len(target)
    z = z_function(nums + [0] + target)
    prefix = [0] * m
    for j in range(1, m):
        prefix[j] = j if target[j - 1] < target[j] else prefix[j - 1]
    print(f"LCP = {z[n + 1:]}\nprefix = {prefix}")
    flags = same_multiset_flags(nums, target)
    for s in range(m - n + 1):
        e = s + n - 1
        max_i = min(z[n + 1 + s], n - 1)
        min_i = max(prefix[e] - s, 0)
        res = max(0, max_i - min_i + 1) if flags[s] else 0
        print(f"s = {s}: cửa sổ {target[s:e + 1]}, cùng đa tập = {flags[s]!s:5}, "
              f"maxI = {max_i}, minI = {min_i} -> {res}")


trace_fast([5, 6, 5], [5, 6, 5, 5, 6, 5])

LCP = [3, 0, 1, 3, 0, 1]
prefix = [0, 1, 1, 1, 4, 4]
s = 0: cửa sổ [5, 6, 5], cùng đa tập = True , maxI = 2, minI = 1 -> 2
s = 1: cửa sổ [6, 5, 5], cùng đa tập = True , maxI = 0, minI = 0 -> 1
s = 2: cửa sổ [5, 5, 6], cùng đa tập = True , maxI = 1, minI = 2 -> 0
s = 3: cửa sổ [5, 6, 5], cùng đa tập = True , maxI = 2, minI = 1 -> 2


# 10. Phiên bản đúng format LeetCode

In [12]:
from collections import Counter
from typing import List


class Solution:
    def countSuffix(self, nums: List[int], target: List[int]) -> List[int]:
        n, m = len(nums), len(target)
        S = nums + [0] + target
        L = len(S)
        z = [0] * L
        l = r = 0
        for i in range(1, L):
            if i < r:
                z[i] = min(r - i, z[i - l])
            while i + z[i] < L and S[z[i]] == S[i + z[i]]:
                z[i] += 1
            if i + z[i] > r:
                l, r = i, i + z[i]

        prefix = [0] * m
        for j in range(1, m):
            prefix[j] = j if target[j - 1] < target[j] else prefix[j - 1]

        cnt = Counter(nums)
        diff = len(cnt)
        ans = []
        for i, v in enumerate(target):
            before = cnt[v]
            cnt[v] -= 1
            if before == 0:
                diff += 1
            elif cnt[v] == 0:
                diff -= 1
            s = i - n + 1
            if s < 0:
                continue
            if diff == 0:
                max_i = min(z[n + 1 + s], n - 1)
                min_i = max(prefix[i] - s, 0)
                ans.append(max(0, max_i - min_i + 1))
            else:
                ans.append(0)
            w = target[s]
            before = cnt[w]
            cnt[w] += 1
            if before == 0:
                diff += 1
            elif cnt[w] == 0:
                diff -= 1
        return ans

In [13]:
sol = Solution()
tests = [
    ([2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2], [1, 1, 0, 0, 0, 1], "ví dụ 1"),
    ([5, 6, 5], [5, 6, 5, 5, 6, 5], [2, 1, 0, 2], "ví dụ 2"),
    ([7], [7, 8, 7], [1, 0, 1], "n = 1"),
    ([3, 3, 3], [3, 3, 3], [3], "toàn bằng nhau: mọi i"),
    ([1, 2, 3], [3, 2, 1, 3, 2, 1], [1, 0, 1, 1], "cửa sổ [1,3,2] khớp i = 1"),
]
for nums, target, expected, name in tests:
    result = sol.countSuffix(nums, target)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:24} -> {result}")

PASS | ví dụ 1                  -> [1, 1, 0, 0, 0, 1]
PASS | ví dụ 2                  -> [2, 1, 0, 2]
PASS | n = 1                    -> [1, 0, 1]
PASS | toàn bằng nhau: mọi i    -> [3]
PASS | cửa sổ [1,3,2] khớp i = 1 -> [1, 0, 1, 1]


In [14]:
import time

random.seed(4109)
n_big, m_big = 5 * 10**4, 10**5
nums_big = [random.randint(1, 3) for _ in range(n_big)]
target_big = [random.randint(1, 3) for _ in range(m_big)]
for k in (0, m_big - n_big):                                  # cài hai cửa sổ khớp ở hai đầu
    i = random.randint(0, n_big - 1)
    target_big[k:k + n_big] = nums_big[:i] + sorted(nums_big[i:], reverse=True)
t0 = time.perf_counter()
res = sol.countSuffix(nums_big, target_big)
print(f"n = {n_big}, m = {m_big}: {sum(1 for x in res if x)} cửa sổ có đáp án > 0, "
      f"chạy {time.perf_counter() - t0:.2f}s")

n = 50000, m = 100000: 2 cửa sổ có đáp án > 0, chạy 0.05s


# 11. Vì sao đúng?

1. **Từng cửa sổ là bài I** (mục 2) — đã chứng minh ở notebook 4107.
2. **Cùng đa tập:** `cnt[v]` luôn bằng (số lần `v` trong `nums`) − (số lần `v` trong cửa sổ). `diff` đếm đúng số khóa khác 0 vì mỗi lần thay đổi ta kiểm tra trạng thái trước/sau. Cùng đa tập ⇔ mọi `cnt` bằng 0 ⇔ `diff == 0`.
3. **minI:** chỗ đi lên trong cửa sổ `[s, e]` là các `j` với `s < j <= e`. `prefix[e]` là chỗ đi lên lớn nhất `<= e`; nếu nó `> s` thì nằm trong cửa sổ, ngược lại cửa sổ không có chỗ đi lên → `minI = max(prefix[e] - s, 0)`.
4. **maxI:** với lính canh `0` không xuất hiện trong dữ liệu, `z[n+1+s]` đúng bằng LCP của `nums` và `target[s:]`, và không vượt `n`. Chặn ở `n - 1` vì chỉ số thí nghiệm `i <= n - 1`.

---

## Trực giác

> Bài I trả lời một câu hỏi bằng ba con số. Bài II hỏi cùng câu hỏi cho `m - n + 1` cửa sổ — nên ta tìm cách để **ba con số ấy "trượt" theo cửa sổ** thay vì tính lại từ đầu.

# 12. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | mọi cửa sổ × mọi thí nghiệm | `O((m-n+1) · n² log n)` | `O(n)` |
| 2. Tra bảng | tính trước `R_i`, tra `tuple` | `O(n² log n + m·n)` | `O(n²)` |
| 3. Bài I từng cửa sổ | `solve_one_window` | `O(n · (m-n+1))` | `O(n)` |
| 4. Ba công cụ | cửa sổ trượt + prefix + hàm Z | `O(n + m)` | `O(n + m)` |

# 13. Đối chiếu bốn cách trên dữ liệu ngẫu nhiên

In [15]:
random.seed(2026)
for _ in range(2000):
    n = random.randint(1, 5)
    m = random.randint(n, 10)
    nums = [random.randint(1, 3) for _ in range(n)]
    target = [random.randint(1, 3) for _ in range(m)]
    if random.random() < 0.5:
        k = random.randint(0, m - n)
        i = random.randint(0, n - 1)
        target[k:k + n] = nums[:i] + sorted(nums[i:], reverse=True)
    expected = count_suffix_bruteforce(nums, target)
    assert count_suffix_lookup(nums, target) == expected
    assert count_suffix_per_window(nums, target) == expected
    assert count_suffix_fast(nums, target) == expected
    assert sol.countSuffix(nums, target) == expected
print("2000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!")

2000 bộ test ngẫu nhiên: 5 hàm cho cùng kết quả!


# 14. Những lỗi dễ mắc

## Lỗi 1 — Lính canh trùng với một giá trị có thật

Nếu dùng lính canh là một số có thể xuất hiện trong dữ liệu (ví dụ `1`), đoạn trùng khớp có thể "chạy xuyên" qua lính canh và LCP bị tính sai (lớn hơn `n`, hoặc khớp nhầm). Ví dụ dưới dùng lính canh `2` — trùng với giá trị trong dữ liệu:

In [16]:
nums_demo, target_demo = [2], [2, 2]
z_bad = z_function(nums_demo + [2] + target_demo)       # SAI: lính canh 2 có trong dữ liệu
z_good = z_function(nums_demo + [0] + target_demo)
print("lính canh 2:", z_bad, "| lính canh 0:", z_good)

lính canh 2: [4, 3, 2, 1] | lính canh 0: [4, 0, 1, 1]


Với lính canh `2`, `z` tại các vị trí của `target` lên tới `2` — lớn hơn `n = 1`, tức là "khớp" cả phần không thuộc `nums`. Nhờ có `min(..., n - 1)` nên bài này may mắn không sai kết quả, nhưng trong các bài khác (đếm số lần xuất hiện của mẫu) đây là lỗi chết người. Luôn chọn lính canh **chắc chắn không xuất hiện**.

## Lỗi 2 — Quên chặn `maxI` ở `n - 1`

Khi cửa sổ trùng hệt `nums`, LCP bằng `n` — nhưng chỉ có các thí nghiệm `i = 0..n-1`.

In [17]:
def wrong_no_cap(nums, target):
    n, m = len(nums), len(target)
    z = z_function(nums + [0] + target)
    prefix = [0] * m
    for j in range(1, m):
        prefix[j] = j if target[j - 1] < target[j] else prefix[j - 1]
    flags = same_multiset_flags(nums, target)
    return [max(0, z[n + 1 + s] - max(prefix[s + n - 1] - s, 0) + 1) if flags[s] else 0
            for s in range(m - n + 1)]


print("Sai :", wrong_no_cap([3, 3, 3], [3, 3, 3]))
print("Đúng:", sol.countSuffix([3, 3, 3], [3, 3, 3]))

Sai : [4]
Đúng: [3]


## Lỗi 3 — Quên `max(..., 0)` khi tính `minI`

`prefix[e] - s` có thể **âm** (chỗ đi lên gần nhất nằm bên trái cửa sổ). Nếu không chặn ở `0`, `minI` âm làm đáp án bị cộng thừa.

In [18]:
def wrong_no_clamp(nums, target):
    n, m = len(nums), len(target)
    z = z_function(nums + [0] + target)
    prefix = [0] * m
    for j in range(1, m):
        prefix[j] = j if target[j - 1] < target[j] else prefix[j - 1]
    flags = same_multiset_flags(nums, target)
    return [max(0, min(z[n + 1 + s], n - 1) - (prefix[s + n - 1] - s) + 1) if flags[s] else 0
            for s in range(m - n + 1)]


print("Sai :", wrong_no_clamp([1], [1, 1]))
print("Đúng:", sol.countSuffix([1], [1, 1]))

Sai : [1, 2]
Đúng: [1, 1]


Ở cửa sổ `s = 1`: `prefix[1] = 0` (không có chỗ đi lên nào), nên `prefix[e] - s = -1`. Code sai lấy `minI = -1` và đếm thành `0 - (-1) + 1 = 2` thí nghiệm, trong khi `nums` chỉ có **một** thí nghiệm.

## Lỗi 4 — So sánh `Counter` cho từng cửa sổ

`Counter(window) == Counter(nums)` tốn `O(n)` mỗi cửa sổ → `O(n·m)`. Phải dùng cửa sổ trượt với biến `diff`.

## Lỗi 5 — Cập nhật `diff` sai khi giá trị đi qua 0 theo chiều âm

`cnt` có thể âm (cửa sổ chứa nhiều hơn `nums`). Cập nhật đúng: nếu **trước** khi đổi `cnt == 0` thì `diff += 1`; nếu **sau** khi đổi `cnt == 0` thì `diff -= 1`. Viết kiểu "nếu `cnt > 0` thì..." sẽ sai khi `cnt` âm.

# 15. Mẹo nhận diện

1. Một bài "một mảng" quen thuộc được nâng thành **"cho mọi cửa sổ trượt"** → giữ nguyên công thức, tìm cách **cập nhật từng đại lượng theo cửa sổ**.
2. **"Cùng đa tập"** trên cửa sổ trượt → bảng đếm + biến `diff` (giống bài *Find All Anagrams in a String*, LeetCode 438).
3. **"Độ dài tiền tố chung của mẫu với mọi vị trí"** → hàm Z (hoặc KMP, hoặc băm + tìm kiếm nhị phân).
4. Thông tin kiểu "**chỗ X gần nhất về bên trái**" cho mọi vị trí → một mảng `prefix` duyệt một lần.

Các bài liên quan: *Find All Anagrams in a String* (438), *Sum of Scores of Built Strings* (2223 — chính là tổng mảng Z), *Find the Index of the First Occurrence in a String* (28).

# 16. Biến thể

## Biến thể 1 — Sắp hậu tố tăng dần

Chỉ thay "chỗ đi lên" bằng "chỗ đi xuống" (`target[j-1] > target[j]`) trong mảng `prefix`.

In [19]:
def count_suffix_ascending(nums: List[int], target: List[int]) -> List[int]:
    n, m = len(nums), len(target)
    z = z_function(nums + [0] + target)
    prefix = [0] * m
    for j in range(1, m):
        prefix[j] = j if target[j - 1] > target[j] else prefix[j - 1]      # chỗ đi xuống
    flags = same_multiset_flags(nums, target)
    return [max(0, min(z[n + 1 + s], n - 1) - max(prefix[s + n - 1] - s, 0) + 1) if flags[s] else 0
            for s in range(m - n + 1)]


def brute_ascending(nums, target):
    n = len(nums)
    res = [nums[:i] + sorted(nums[i:]) for i in range(n)]
    return [sum(r == target[k:k + n] for r in res) for k in range(len(target) - n + 1)]


random.seed(3)
for _ in range(1000):
    n = random.randint(1, 4); m = random.randint(n, 8)
    a = [random.randint(1, 3) for _ in range(n)]
    t = [random.randint(1, 3) for _ in range(m)]
    assert count_suffix_ascending(a, t) == brute_ascending(a, t)
print(count_suffix_ascending([3, 1, 2], [3, 1, 2, 1, 2, 3]), "| khớp brute force trên 1000 test")

[2, 0, 0, 1] | khớp brute force trên 1000 test


## Biến thể 2 — Đếm số cửa sổ là hoán vị của `nums`

Chỉ cần công cụ 1: đếm số cửa sổ có `diff == 0`.

In [20]:
print(sum(same_multiset_flags([1, 2], [2, 1, 1, 2, 2])))   # cửa sổ [2,1], [1,2] -> 2

2


# 17. Template tư duy

```python
# 1. Hàm Z trên nums + [SENTINEL] + target  -> LCP với mọi vị trí
# 2. prefix[j] = "chỗ đặc biệt" gần nhất <= j
# 3. Cửa sổ trượt: cnt + diff
for i, v in enumerate(target):
    add(v)                         # v vào
    s = i - n + 1
    if s < 0: continue
    ans.append(f(diff == 0, z[n + 1 + s], prefix[i], s))
    remove(target[s])              # target[s] ra
```

# 18. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`nums = [3, 1, 3]`, `target = [3, 3, 1, 3, 1, 3]`. Lập bảng `s | cửa sổ | cùng đa tập | LCP | maxI | prefix[e] | minI | ans`. Kiểm tra bằng cell dưới.

In [21]:
# Kiểm tra Bài 1
trace_fast([3, 1, 3], [3, 3, 1, 3, 1, 3])

LCP = [1, 3, 0, 3, 0, 1]
prefix = [0, 0, 0, 3, 3, 5]
s = 0: cửa sổ [3, 3, 1], cùng đa tập = True , maxI = 1, minI = 0 -> 2
s = 1: cửa sổ [3, 1, 3], cùng đa tập = True , maxI = 2, minI = 2 -> 1
s = 2: cửa sổ [1, 3, 1], cùng đa tập = False, maxI = 0, minI = 1 -> 0
s = 3: cửa sổ [3, 1, 3], cùng đa tập = True , maxI = 2, minI = 2 -> 1


## Bài 2 — Dự đoán output

```python
print(z_function("aaaa"))
print(z_function("abab"))
print(z_function([1, 2, 0, 1, 2, 1]))
```

In [22]:
print(z_function("aaaa"))
print(z_function("abab"))
print(z_function([1, 2, 0, 1, 2, 1]))

[4, 3, 2, 1]
[4, 0, 2, 0]
[6, 0, 0, 2, 0, 1]


## Bài 3 — Tự viết hàm Z

Viết lại hàm Z `O(L)` không nhìn code mẫu.

In [23]:
def exercise_3_z(S) -> List[int]:
    # TODO: hộp [l, r), khởi tạo z[i] = min(r - i, z[i - l]) khi i < r, rồi kéo dài
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_z("aabxaab") == [7, 1, 0, 0, 3, 1, 0]

## Bài 4 — Find All Anagrams (LeetCode 438)

Trả về danh sách vị trí bắt đầu `s` sao cho `text[s : s + len(p)]` là hoán vị của `p`.

```python
find_anagrams("cbaebabacd", "abc")   -> [0, 6]
find_anagrams("abab", "ab")          -> [0, 1, 2]
```

In [24]:
def exercise_4_find_anagrams(text: str, p: str) -> List[int]:
    # TODO
    raise NotImplementedError


# assert exercise_4_find_anagrams("cbaebabacd", "abc") == [0, 6]
# assert exercise_4_find_anagrams("abab", "ab") == [0, 1, 2]

## Bài 5 — Số lần xuất hiện của mẫu bằng hàm Z

Đếm số vị trí `s` mà `pattern` xuất hiện trong `text` bắt đầu tại `s`, dùng hàm Z.

```python
count_occurrences("abababa", "aba")   -> 3
count_occurrences("aaaaa", "aa")      -> 4
```

In [25]:
def exercise_5_count_occurrences(text: str, pattern: str) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_count_occurrences("abababa", "aba") == 3
# assert exercise_5_count_occurrences("aaaaa", "aa") == 4

## Bài 6 — Mini challenge: LCP bằng băm + tìm kiếm nhị phân

Viết hàm trả về `LCP(nums, target[s:])` cho mọi `s`, **không dùng hàm Z**, mà dùng **băm đa thức (polynomial rolling hash)** + **tìm kiếm nhị phân** trên độ dài. Kiểm tra bằng cách so với hàm Z.

Gợi ý: `h[i]` = băm của `i` phần tử đầu; băm của đoạn `[a, b)` là `h[b] - h[a] · B^(b-a)` (mod một số nguyên tố lớn).

In [26]:
def exercise_6_lcp_hash(nums: List[int], target: List[int]) -> List[int]:
    # TODO
    raise NotImplementedError


# nums_t, target_t = [2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]
# assert exercise_6_lcp_hash(nums_t, target_t) == z_function(nums_t + [0] + target_t)[4:]

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `z[0] = L`, `l = r = 0`.
- Mức 2: `if i < r: z[i] = min(r - i, z[i - l])`.
- Mức 3: kéo dài bằng `while`, rồi `if i + z[i] > r: l, r = i, i + z[i]`.

**Bài 4**

- Mức 1: cửa sổ trượt độ dài `len(p)` với `cnt` và `diff`.
- Mức 2: ký tự vào giảm `cnt`, ký tự ra tăng `cnt`.
- Mức 3: ghi `s` khi `diff == 0`.

**Bài 5**

- Mức 1: ghép `pattern + "#" + text` (`#` không có trong chuỗi).
- Mức 2: vị trí của `text[s]` trong chuỗi ghép là `len(pattern) + 1 + s`.
- Mức 3: đếm các `s` có `z == len(pattern)`.

**Bài 6**

- Mức 1: tính `h1` cho `nums`, `h2` cho `target`, và mảng lũy thừa `pw`.
- Mức 2: với mỗi `s`, tìm nhị phân độ dài lớn nhất `L <= min(n, m - s)` mà băm hai đoạn bằng nhau.
- Mức 3: "bằng nhau ở độ dài `L`" là tính chất đơn điệu (đúng với `L` thì đúng với mọi độ dài nhỏ hơn) nên tìm kiếm nhị phân hợp lệ.

# 19. Đáp án tham khảo

## Đáp án Bài 1

`LCP = [1, 3, 0, 3, 0, 1]`, `prefix = [0, 0, 0, 3, 3, 5]`:

| s | cửa sổ | cùng đa tập | LCP | maxI | prefix[e] | minI | ans |
|---:|---|:---:|---:|---:|---:|---:|---:|
| 0 | `[3,3,1]` | ✔ | 1 | 1 | 0 | 0 | 2 |
| 1 | `[3,1,3]` | ✔ | 3 | 2 | 3 | 2 | 1 |
| 2 | `[1,3,1]` | ✗ | 0 | 0 | 3 | 1 | 0 |
| 3 | `[3,1,3]` | ✔ | 3 | 2 | 5 | 2 | 1 |

Kết quả `[2, 1, 0, 1]`. Cửa sổ 0 `[3, 3, 1]` khớp cả `R_0` (sắp cả mảng) và `R_1` (giữ `3`, sắp `[1, 3]` thành `[3, 1]`).

## Đáp án Bài 2

```text
[4, 3, 2, 1]
[4, 0, 2, 0]
[6, 0, 0, 2, 0, 1]
```

In [27]:
# Đáp án Bài 3
def answer_3_z(S) -> List[int]:
    L = len(S)
    z = [0] * L
    if L:
        z[0] = L
    l = r = 0
    for i in range(1, L):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < L and S[z[i]] == S[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z


assert answer_3_z("aabxaab") == [7, 1, 0, 0, 3, 1, 0]
print("Bài 3: PASS")

Bài 3: PASS


In [28]:
# Đáp án Bài 4
def answer_4_find_anagrams(text: str, p: str) -> List[int]:
    k = len(p)
    cnt = Counter(p)
    diff = len(cnt)
    res = []

    def change(ch, d):
        nonlocal diff
        before = cnt[ch]
        cnt[ch] += d
        if before == 0:
            diff += 1
        elif cnt[ch] == 0:
            diff -= 1

    for i, ch in enumerate(text):
        change(ch, -1)
        s = i - k + 1
        if s < 0:
            continue
        if diff == 0:
            res.append(s)
        change(text[s], +1)
    return res


assert answer_4_find_anagrams("cbaebabacd", "abc") == [0, 6]
assert answer_4_find_anagrams("abab", "ab") == [0, 1, 2]
print("Bài 4: PASS")

Bài 4: PASS


In [29]:
# Đáp án Bài 5
def answer_5_count_occurrences(text: str, pattern: str) -> int:
    z = z_function(pattern + "#" + text)
    k = len(pattern)
    return sum(1 for s in range(len(text)) if z[k + 1 + s] == k)


assert answer_5_count_occurrences("abababa", "aba") == 3
assert answer_5_count_occurrences("aaaaa", "aa") == 4
print("Bài 5: PASS")

Bài 5: PASS


In [30]:
# Đáp án Bài 6
def answer_6_lcp_hash(nums: List[int], target: List[int]) -> List[int]:
    MOD, B = (1 << 61) - 1, 1_000_003
    n, m = len(nums), len(target)
    pw = [1] * (max(n, m) + 1)
    for i in range(1, len(pw)):
        pw[i] = pw[i - 1] * B % MOD

    def build(a):
        h = [0] * (len(a) + 1)
        for i, v in enumerate(a):
            h[i + 1] = (h[i] * B + v) % MOD
        return h

    h1, h2 = build(nums), build(target)
    seg = lambda h, a, b: (h[b] - h[a] * pw[b - a]) % MOD
    res = []
    for s in range(m):
        lo, hi = 0, min(n, m - s)
        while lo < hi:                          # độ dài lớn nhất mà băm khớp
            mid = (lo + hi + 1) // 2
            if seg(h1, 0, mid) == seg(h2, s, s + mid):
                lo = mid
            else:
                hi = mid - 1
        res.append(lo)
    return res


nums_t, target_t = [2, 2, 3], [2, 2, 3, 2, 1, 3, 2, 2]
assert answer_6_lcp_hash(nums_t, target_t) == z_function(nums_t + [0] + target_t)[4:]
random.seed(6)
for _ in range(300):
    a = [random.randint(1, 2) for _ in range(random.randint(1, 6))]
    t = [random.randint(1, 2) for _ in range(random.randint(len(a), 12))]
    assert answer_6_lcp_hash(a, t) == z_function(a + [0] + t)[len(a) + 1:]
print("Bài 6: PASS (khớp hàm Z trên 300 test)")

Bài 6: PASS (khớp hàm Z trên 300 test)


# 20. Tổng kết

**Count Valid Suffix Sorts II** = bài I + ba công cụ "trượt":

| Đại lượng của bài I | Công cụ cho mọi cửa sổ |
|---|---|
| Cùng đa tập | cửa sổ trượt `cnt` + `diff` |
| `minI` (chỗ đi lên cuối cùng) | mảng `prefix` + `max(prefix[e] - s, 0)` |
| `maxI` (tiền tố chung) | hàm Z trên `nums + [0] + target`, chặn ở `n - 1` |

## Công thức nhớ nhanh

```text
ans[s] = 0                                                   nếu khác đa tập
ans[s] = max(0, min(z[n+1+s], n-1) - max(prefix[e]-s, 0) + 1)  ngược lại
```

## Độ phức tạp

```text
Time:  O(n + m)
Space: O(n + m)
```

# 21. Checklist trước khi submit

- [ ] Lính canh trong hàm Z **không trùng** giá trị nào của dữ liệu (`0` vì giá trị `>= 1`)?
- [ ] `maxI = min(z[...], n - 1)`?
- [ ] `minI = max(prefix[e] - s, 0)` (chỗ đi lên tại đúng `s` không tính)?
- [ ] Chỗ đi lên dùng `<` nghiêm ngặt (bằng nhau vẫn là không tăng)?
- [ ] `diff` cập nhật đúng cả khi `cnt` âm?
- [ ] Phần tử ra khỏi cửa sổ được cập nhật **sau** khi ghi đáp án?
- [ ] Đã test `n = 1`, `n = m`, cửa sổ trùng hệt `nums`?